# CASMI 2026 — Phase 2: Expanded Adducts + Analog Expansion

**Changes from Phase 1:**
1. Expanded adduct table: dimers ([2M+H]+ etc.) now correctly converted (were delta=0 before)
2. Analog expansion: for top-3 library hits per molecule, add 10 most fingerprint-similar
   structures each (SMILES n-gram fingerprints, pure Python — no RDKit needed).
   Catches close analogs missed by exact-mass search.

Expected: modest LB lift (0.129 → ~0.15-0.18). The big novel-structure gains need Phase 3.


In [ ]:

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import os, time, glob
from collections import defaultdict
from hashlib import md5
print("imports ok")


In [ ]:

# ================= Core (Phase 1 + expanded adducts) =================
ELECTRON = 0.00054858
PROTON = 1.00782503223 - ELECTRON
NA_ION = 22.9897692809 - ELECTRON
K_ION = 38.9637074864 - ELECTRON
NH4_ION = 18.033823
H2O = 18.010564683
CL_ION = 34.968852682 + ELECTRON
FORMATE_ADDUCT = 46.005480

ADDUCT_DELTA_P1 = {
    "[M+H]+": -PROTON, "[M+NH4]+": -NH4_ION,
    "[M-H2O+H]+": -(PROTON - H2O), "[M-2H2O+H]+": -(PROTON - 2*H2O),
    "[M+Na]+": -NA_ION, "[M+K]+": -K_ION,
    "[M-H]-": PROTON, "[M-H2O-H]-": PROTON + H2O,
    "[M+CH2O2-H]-": -(FORMATE_ADDUCT - PROTON), "[M+Cl]-": -CL_ION,
}
EXPANDED = {ad: (1, d) for ad, d in ADDUCT_DELTA_P1.items()}
EXPANDED.update({
    "[2M+H]+": (2, -PROTON), "[2M+Na]+": (2, -NA_ION), "[2M+K]+": (2, -K_ION),
    "[2M+NH4]+": (2, -NH4_ION), "[2M-H]-": (2, PROTON),
})

def neutral_mass(precursor_mz, adduct):
    precursor_mz = np.asarray(precursor_mz, dtype=np.float64)
    scalar = precursor_mz.ndim == 0
    precursor_mz = np.atleast_1d(precursor_mz)
    if isinstance(adduct, str):
        adduct = [adduct] * len(precursor_mz)
    out = np.full(len(precursor_mz), np.nan)
    for i, (pm, ad) in enumerate(zip(precursor_mz, adduct)):
        e = EXPANDED.get(ad)
        if e is not None:
            mult, delta = e
            out[i] = (pm + delta) / mult
    return float(out[0]) if scalar and len(out) == 1 else out

def preprocess_peaks(mzs, intensities, precursor_mz, min_rel_int=0.01, max_mz_over_precursor=2.0):
    mzs = np.asarray(mzs, dtype=np.float64); intensities = np.asarray(intensities, dtype=np.float64)
    if len(mzs) == 0: return mzs, intensities
    o = np.argsort(mzs); mzs, intensities = mzs[o], intensities[o]
    base = intensities.max()
    if base <= 0: return np.empty(0), np.empty(0)
    keep = (intensities >= min_rel_int * base) & (mzs <= precursor_mz + max_mz_over_precursor)
    mzs, intensities = mzs[keep], intensities[keep]
    if len(mzs) == 0: return mzs, intensities
    intensities = np.sqrt(intensities); intensities /= np.linalg.norm(intensities)
    return mzs, intensities

def entropy_similarity(mzs_a, int_a, mzs_b, int_b, fragment_tol=0.01):
    mzs_a = np.asarray(mzs_a, dtype=np.float64); int_a = np.asarray(int_a, dtype=np.float64)
    mzs_b = np.asarray(mzs_b, dtype=np.float64); int_b = np.asarray(int_b, dtype=np.float64)
    if len(mzs_a) == 0 or len(mzs_b) == 0: return 0.0
    pa = int_a / int_a.sum(); pb = int_b / int_b.sum()
    s_a = float(-np.sum(pa * np.log(pa + 1e-300))); s_b = float(-np.sum(pb * np.log(pb + 1e-300)))
    i = j = 0; na, nb = len(mzs_a), len(mzs_b); merged = []
    while i < na and j < nb:
        dm = mzs_a[i] - mzs_b[j]
        if abs(dm) <= fragment_tol: merged.append(pa[i] + pb[j]); i += 1; j += 1
        elif dm < 0: merged.append(pa[i]); i += 1
        else: merged.append(pb[j]); j += 1
    while i < na: merged.append(pa[i]); i += 1
    while j < nb: merged.append(pb[j]); j += 1
    merged = np.array(merged); merged /= merged.sum()
    s_ab = float(-np.sum(merged * np.log(merged + 1e-300)))
    return float(np.clip(1.0 - (2.0*s_ab - s_a - s_b) / np.log(4.0), 0.0, 1.0))

def rrf_fuse(ranked_lists, k=60):
    scores = {}
    for lst in ranked_lists:
        for rank, cid in enumerate(lst):
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank)
    return scores

def ppm_window_indices(sorted_masses, query_mass, ppm):
    tol = query_mass * ppm * 1e-6
    lo = np.searchsorted(sorted_masses, query_mass - tol, side="left")
    hi = np.searchsorted(sorted_masses, query_mass + tol, side="right")
    return lo, hi
print("core loaded")


In [ ]:

cands = glob.glob("/kaggle/input/**/train.parquet", recursive=True)
assert cands, "attach competition data via + Add Input"
INPUT_DIR = os.path.dirname(cands[0])
t0 = time.time()
train = pq.read_table(os.path.join(INPUT_DIR, "train.parquet")).to_pandas()
test = pq.read_table(os.path.join(INPUT_DIR, "test.parquet")).to_pandas()
print(f"loaded {time.time()-t0:.0f}s; train {train.shape}, test {test.shape}")


In [ ]:

t0 = time.time()
n = len(train)
tr_mz = train["ms2_mzs"].to_numpy(); tr_int = train["ms2_normalized_intensities"].to_numpy()
tr_prec = train["precursor_mz"].to_numpy(dtype=np.float64)
tr_adduct = train["adduct"].to_numpy(); tr_key14 = train["inchikey14"].to_numpy()
tr_smiles = train["normalized_smiles"].to_numpy()
np_mask = (train["ingest_lib"] == "enveda-np-examples").to_numpy()

tr_neutral = neutral_mass(tr_prec, tr_adduct)
valid = ~np.isnan(tr_neutral)
print(f"valid adducts: {valid.sum()}/{n} ({100*valid.sum()/n:.1f}%)")
kept = np.nonzero(valid)[0]

lib_mzs, lib_int, lib_neu, lib_key = [], [], [], []
for i in kept:
    pm, pi = preprocess_peaks(tr_mz[i], tr_int[i], tr_prec[i])
    lib_mzs.append(pm); lib_int.append(pi); lib_neu.append(tr_neutral[i]); lib_key.append(tr_key14[i])
    if len(lib_mzs) % 500000 == 0: print(f"  {len(lib_mzs)} ({time.time()-t0:.0f}s)")
lib_mzs = np.array(lib_mzs, dtype=object); lib_int = np.array(lib_int, dtype=object)
lib_neu = np.array(lib_neu); lib_key = np.array(lib_key)
del train, tr_mz, tr_int, tr_neutral
import gc; gc.collect()
order = np.argsort(lib_neu); lib_neu_sorted = lib_neu[order]; lib_pos2row = order
del lib_neu, order; gc.collect()

key_to_smiles = {}
for k, s in zip(lib_key, tr_smiles[kept]):
    if k not in key_to_smiles: key_to_smiles[k] = s
print(f"index: {len(key_to_smiles)} structures, {time.time()-t0:.0f}s")


In [ ]:

# ============ SMILES n-gram fingerprints (pure Python, no RDKit) ============
# Character 4-grams of SMILES hashed to 2048 bits. Correlates with structural
# similarity; sufficient for analog expansion. Works offline.
t0 = time.time()
keys = list(key_to_smiles.keys())
NK = len(keys)
fp_packed = np.zeros((NK, 256), dtype=np.uint8)
for j, k in enumerate(keys):
    s = key_to_smiles[k]
    # 3-grams and 4-grams
    for kk in (3, 4):
        for i in range(len(s) - kk + 1):
            ng = s[i:i+kk]
            h = int(md5(ng.encode()).hexdigest(), 16) % 2048
            fp_packed[j, h // 8] |= (1 << (h % 8))
    if (j+1) % 50000 == 0: print(f"  {j+1}/{NK} ({time.time()-t0:.0f}s)")
key_to_idx = {k: j for j, k in enumerate(keys)}
POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint16)
fp_bitsum = POP[fp_packed].sum(axis=1)
print(f"fingerprints: {time.time()-t0:.0f}s")

def topn_similar(query_key, n=10):
    # Top-n most fingerprint-similar structures to query_key (excluding self).
    qi = key_to_idx.get(query_key)
    if qi is None: return []
    q = fp_packed[qi]
    inter = POP[np.bitwise_and(fp_packed, q)].sum(axis=1)
    union = fp_bitsum[qi] + fp_bitsum - inter
    sim = np.divide(inter, union, out=np.zeros(NK), where=union > 0)
    sim[qi] = -1
    top = np.argpartition(-sim, n)[:n]
    top = top[np.argsort(-sim[top])]
    return [keys[i] for i in top]


In [ ]:

# ================= Search =================
def library_search(q_mzs, q_int, q_neutral, topk=100):
    lo, hi = ppm_window_indices(lib_neu_sorted, q_neutral, 10.0)
    best = {}
    for pos in range(lo, hi):
        row = lib_pos2row[pos]
        s = entropy_similarity(q_mzs, q_int, lib_mzs[row], lib_int[row])
        if s <= 0: continue
        k = lib_key[row]
        if s > best.get(k, 0.0): best[k] = s
    return sorted(best.items(), key=lambda kv: -kv[1])[:topk]

def predict_molecule(spectra):
    per_spec = []; neutrals = []
    for mzs, inten, prec, ad in spectra:
        qm, qi = preprocess_peaks(mzs, inten, prec)
        qn = neutral_mass(prec, ad)
        if np.isnan(qn): continue
        qn = float(qn); neutrals.append(qn)
        if len(qm) == 0: continue
        per_spec.append([k for k, _ in library_search(qm, qi, qn)])
    if not per_spec: return [], 0.0
    scores = rrf_fuse(per_spec, k=60)
    lib_order = sorted(scores, key=lambda k: -scores[k])
    # analog expansion: top-3 hits -> 10 most similar each
    expanded = list(lib_order)
    seen = set(expanded)
    for k in lib_order[:3]:
        for ak in topn_similar(k, n=10):
            if ak not in seen:
                seen.add(ak); expanded.append(ak)
    qn = float(np.median(neutrals)) if neutrals else 0.0
    return expanded, qn

def fill_to_25(ranked, q_neutral):
    out, seen = [], set()
    for k in ranked:
        if k not in seen: seen.add(k); out.append(k)
        if len(out) >= 25: return out
    for ppm in (500.0, 5000.0, 50000.0):
        lo, hi = ppm_window_indices(lib_neu_sorted, q_neutral, ppm)
        mid = (lo+hi)//2; a, b = mid-1, mid
        while (a >= lo or b < hi) and len(out) < 25:
            if a >= lo:
                k = lib_key[lib_pos2row[a]]
                if k not in seen: seen.add(k); out.append(k)
                a -= 1
            if b < hi and len(out) < 25:
                k = lib_key[lib_pos2row[b]]
                if k not in seen: seen.add(k); out.append(k)
                b += 1
        if len(out) >= 25: break
    return out
print("search ready")


In [ ]:

RUN_VALIDATION = True
if RUN_VALIDATION:
    t0 = time.time()
    orig_to_comp = {o: c for c, o in enumerate(kept)}
    np_comp = set(orig_to_comp[o] for o in np.nonzero(np_mask)[0] if o in orig_to_comp)
    np_by_mol = defaultdict(list)
    for cr in np_comp:
        np_by_mol[lib_key[cr]].append(cr)
    print(f"{len(np_by_mol)} held-out molecules")
    rr_sum, n_mol = 0.0, 0
    for true_k, crows in np_by_mol.items():
        spectra = [(lib_mzs[cr], lib_int[cr], float(tr_prec[kept[cr]]), str(tr_adduct[kept[cr]]))
                   for cr in crows]
        ranked, qn = predict_molecule(spectra)
        ranked = fill_to_25(ranked, qn)
        try: rr_sum += 1.0 / (ranked.index(true_k) + 1)
        except ValueError: pass
        n_mol += 1
        if n_mol % 50 == 0: print(f"  {n_mol} MRR={rr_sum/n_mol:.4f}")
    print(f"VALIDATION MRR@25 (Phase 2): {rr_sum/n_mol:.4f} ({time.time()-t0:.0f}s)")


In [ ]:

t0 = time.time()
te_mz = test["ms2_mzs"].to_numpy(); te_int = test["ms2_normalized_intensities"].to_numpy()
te_prec = test["precursor_mz"].to_numpy(dtype=np.float64)
te_adduct = test["adduct"].to_numpy(); te_mol = test["molecule_id"].to_numpy()
mol_rows = defaultdict(list)
for i, m in enumerate(te_mol): mol_rows[m].append(i)
print(f"{len(mol_rows)} molecules")
del test; import gc; gc.collect()
rows = []
for j, (mol, idxs) in enumerate(mol_rows.items()):
    spectra = [(np.asarray(te_mz[i], dtype=np.float64), np.asarray(te_int[i], dtype=np.float64),
                float(te_prec[i]), str(te_adduct[i])) for i in idxs]
    ranked, qn = predict_molecule(spectra)
    ranked = fill_to_25(ranked, qn)
    rows.append((mol, ";".join(key_to_smiles[k] for k in ranked[:25])))
    if (j+1) % 50 == 0: print(f"  {j+1}/{len(mol_rows)} ({time.time()-t0:.0f}s)")
sub = pd.DataFrame(rows, columns=["molecule_id", "smiles"])
assert sub["molecule_id"].is_unique
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(f"done {sub.shape} in {time.time()-t0:.0f}s")
